# IRIS H100 Verification (Kaggle)

- Change class: `targeted_fix` (validation readiness)
- Target failure categories/signals: `F_SEARCH`, `F_EVAL` (runtime stability and verification path)
- Phase target: `C`

This notebook verifies the current IRIS baseline on Kaggle H100 using local wheels and the packaged repo snapshot.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_INPUT_DIR = Path('/kaggle/input/datasets/danielwuu/iris-main/packages')
WHEEL_DIR = Path('/kaggle/input/datasets/danielwuu/wheels/wheels/linux_cp312')
WORK_REPO = Path('/kaggle/working/iris-main')
ARTIFACT_ROOT = WORK_REPO / 'artifacts' / 'h100_validation'

def run(cmd: str, cwd: Path | None = None, ok: tuple[int, ...] = (0,), env: dict | None = None):
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(result.stdout)
    if result.returncode not in ok:
        raise RuntimeError(f'command failed ({result.returncode}): {cmd}')
    return result

print('python:', sys.version)
print('repo input exists:', REPO_INPUT_DIR.exists(), REPO_INPUT_DIR)
print('wheel dir exists:', WHEEL_DIR.exists(), WHEEL_DIR)


In [ ]:
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('This notebook expects Python 3.12 because wheels are cp312.')

if not REPO_INPUT_DIR.exists():
    raise FileNotFoundError(f'Missing repo input dir: {REPO_INPUT_DIR}')
if not WHEEL_DIR.exists():
    raise FileNotFoundError(f'Missing wheel dir: {WHEEL_DIR}')

if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)
shutil.copytree(REPO_INPUT_DIR, WORK_REPO)
WORK_REPO.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print('copied repo to:', WORK_REPO)
print('artifact root:', ARTIFACT_ROOT)


In [ ]:
run(f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} --upgrade pip")

def wheel_version(path: Path) -> str:
    # wheel naming: <name>-<version>-...
    return path.name.split('-')[1]

jax_whls = sorted(WHEEL_DIR.glob('jax-*.whl'))
jaxlib_whls = sorted(WHEEL_DIR.glob('jaxlib-*.whl'))
if not jax_whls or not jaxlib_whls:
    raise RuntimeError('Missing jax/jaxlib wheels in wheel directory.')

jax_by_version = {wheel_version(p): p for p in jax_whls}
jaxlib_by_version = {wheel_version(p): p for p in jaxlib_whls}
common_versions = sorted(set(jax_by_version) & set(jaxlib_by_version))
if not common_versions:
    raise RuntimeError('No common jax/jaxlib version found in wheels.')

target_version = common_versions[-1]
target_jax = jax_by_version[target_version]
target_jaxlib = jaxlib_by_version[target_version]

plugin_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_plugin-*.whl'))}
pjrt_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_pjrt-*.whl'))}

wheels_to_install = [target_jax, target_jaxlib]
if target_version in plugin_by_version and target_version in pjrt_by_version:
    wheels_to_install.extend([plugin_by_version[target_version], pjrt_by_version[target_version]])
    print('using matched CUDA plugin/pjrt version:', target_version)
else:
    print('no matched plugin/pjrt for version', target_version, '- uninstalling any preinstalled CUDA plugin/pjrt')
    run(f"{sys.executable} -m pip uninstall -y jax_cuda12_plugin jax_cuda12_pjrt", ok=(0, 1))

wheel_args = ' '.join(str(p) for p in wheels_to_install)
run(
    f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} "
    f"--upgrade --force-reinstall --no-deps {wheel_args}"
)

run(
    f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} "
    "--upgrade --force-reinstall numpy pytest flax optax orbax-checkpoint ml-dtypes scipy"
)

run(f"{sys.executable} -m pip show jax jaxlib", ok=(0, 1))


In [ ]:
run('nvidia-smi -L')
run('nvidia-smi')

import jax
import flax  # noqa: F401
import optax  # noqa: F401

print('jax version:', jax.__version__)
print('jax devices:', jax.devices())
if not any(dev.platform == 'gpu' for dev in jax.devices()):
    raise RuntimeError('No JAX GPU device detected on this Kaggle runtime.')


In [ ]:
env = os.environ.copy()
env.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
env.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.85')

run(f"{sys.executable} -m pytest -q", cwd=WORK_REPO, env=env)
run(f"{sys.executable} scripts/s1_smoke.py --device gpu", cwd=WORK_REPO, env=env)
run(f"{sys.executable} scripts/s2_structural.py", cwd=WORK_REPO, env=env)


In [ ]:
train_out = ARTIFACT_ROOT / 'toy_train_gpu_h100'
run(
    f"{sys.executable} scripts/train_toy.py --segments 2 --device gpu --output-dir {train_out}",
    cwd=WORK_REPO,
    env=env,
)
run(
    f"{sys.executable} scripts/eval_toy.py --device gpu --output-dir {train_out}",
    cwd=WORK_REPO,
    env=env,
)


In [ ]:
resume_out = ARTIFACT_ROOT / 'toy_resume_h100'

run(
    (
        f"{sys.executable} scripts/train_toy.py --segments 1 --device gpu "
        f"--output-dir {resume_out} --crash-point pre_commit --crash-segment 0"
    ),
    cwd=WORK_REPO,
    env=env,
    ok=(0, 2),
)

run(
    f"{sys.executable} scripts/train_toy.py --segments 1 --device gpu --output-dir {resume_out}",
    cwd=WORK_REPO,
    env=env,
)

journal_path = resume_out / 'segment_journal.jsonl'
events = [json.loads(line) for line in journal_path.read_text(encoding='utf-8').splitlines() if line.strip()]
applied_segment0 = [
    e for e in events if int(e.get('segment_id', -1)) == 0 and e.get('status') == 'APPLIED'
]
print('journal path:', journal_path)
print('event count:', len(events))
print('applied events for segment 0:', len(applied_segment0))
if len(applied_segment0) != 1:
    raise RuntimeError('Resume exactly-once check failed: segment 0 APPLIED count != 1')

print('PASS: H100 validation finished')
print('train artifacts:', train_out)
print('resume artifacts:', resume_out)


## Expected Outcome

1. JAX detects at least one GPU device.
2. `pytest`, `S1`, and `S2` complete without crash.
3. GPU toy training + eval run completes.
4. Pre-commit crash + resume leaves exactly one `APPLIED` event for segment `0`.


In [ ]:
zip_base = Path('/kaggle/working/iris_h100_validation_outputs')
zip_file = zip_base.with_suffix('.zip')
if zip_file.exists():
    zip_file.unlink()

archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(ARTIFACT_ROOT))
print('compressed artifact root:', ARTIFACT_ROOT)
print('zip output:', archive_path)
print('size bytes:', Path(archive_path).stat().st_size)
